# 강의 05 · 실습 2 — 평가 데이터셋과 채점자 · (3) 변형


## 1. 문제상황

- 시립도서관의 안내 서비스는 FAQ 문서를 근거로 이용자의 질문에 답합니다.
- 담당자는 채점 기준으로 「어떻게 답해야 한다」는 서술 문장 대신 FAQ의 정답 문장 자체를 쓰고 싶어 합니다.
- 평가할 질문은 5개이고, 그중 2개는 FAQ에 없는 질문이라 서비스가 확인할 수 없다고 답해야 합니다.
- 담당자는 모델 채점자 점수의 평균이 0.8에 못 미치면 새 버전의 배포를 막으려 합니다.


## 2. 문제와 목표

- **문제**: 채점 기준이 서술 문장이면 채점자가 정답 원문을 볼 수 없어 맞는 답도 근거 없음으로 판정할 수 있습니다. 배포 차단 기준도 더 엄격해야 합니다.
- **목표**
  - 질문 5개와 정답 텍스트를 짝지은 데이터셋을 등록합니다.
    - 데이터셋 `sesac-lec05-ex02-library`: 질문 5개(FAQ 안 3개 — 인사 1개 포함, 밖 2개)와 정답 텍스트의 짝. 값(`GOLDEN`)과 모델 채점자에 넣는 채점 지시문(`JUDGE_GUIDE`)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
  - 정답 텍스트와의 부합을 판정하는 모델 채점자와 규칙 채점자로 서비스의 답을 자동 채점합니다.
    - 규칙 채점자 `rule_content`: FAQ 밖 질문이면 답에 「확인할 수 없」 표현이 있는지, FAQ 안 질문이면 답이 비어 있지 않은지로 0 또는 1점
    - 모델 채점자 `judge_faithful`: 질문·정답 텍스트·답을 함께 받아 「답이 정답과 부합하고 어긋난 지점이 없는가」로 0 또는 1점. 지시문은 단계 0에 주어져 있습니다
  - 모델 채점자 점수의 평균이 기준 0.8에 못 미치면 차단으로 판정하는 평가 절차를 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 데이터셋의 질문 5개마다 규칙 점수와 모델 점수가 화면에 출력되고,
  - FAQ 밖 질문 2개의 규칙 점수가 1이고,
  - 모델 점수의 평균이 기준 0.8과 비교되어 통과 또는 차단이 마지막 줄에 출력되는 것을 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex02_s3_diagram.svg)


## 4. 단계별 요구사항

1. **골든 데이터셋을 등록합니다.**
    - 질문 5개와 정답 텍스트(`answer`)·FAQ 안 질문 여부(`must_know`)를 짝지어 `sesac-lec05-ex02-library` 데이터셋으로 올립니다.
    - FAQ 안 질문의 정답 텍스트는 FAQ의 답 문장을 그대로 씁니다.
    - FAQ 밖 질문의 정답 텍스트는 「해당 내용은 확인할 수 없습니다.」입니다.
    - 같은 이름의 데이터셋이 이미 있으면 새로 만들지 않고 재사용합니다.
    - 질문과 정답 텍스트는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 단계 ①은 데이터셋을 새로 만들면 「데이터셋 '이름'을 만들고 예시 5개를 올렸습니다」, 이미 있으면 「데이터셋 '이름'이 이미 있어 재사용합니다」 문장을 출력합니다.
2. **평가 대상을 지정합니다.**
    - `target` 함수는 `inputs` 딕셔너리에서 질문을 꺼내 안내 서비스 `answer`를 부르고, 답을 `{"answer": 답}` 딕셔너리로 돌려줍니다.
3. **규칙 채점자를 만듭니다.**
    - `rule_content`는 FAQ 밖 질문이면 답에 「확인할 수 없」 표현이 있는지 보고, FAQ 안 질문이면 답이 비어 있지 않은지 보고, 0 또는 1점을 `{"key": "rule_content", "score": 점수}`로 돌려줍니다.
4. **모델 채점자를 만듭니다.**
    - `Judge` 스키마(`faithful`·`reason`)를 선언하고, 구조화 출력을 붙인 모델에 질문·정답 텍스트·답을 함께 넘겨 「답이 정답과 부합하고 어긋난 지점이 없는가」를 판정하게 하고, `{"key": "judge_faithful", "score": 점수, "comment": 근거}`로 돌려줍니다.
    - 모델에 주는 지시문은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 정의한 뒤 대출 규정 질문(「책은 몇 권까지 며칠 동안 빌릴 수 있나요?」) 하나로 두 채점자를 한 번씩 불러 점수를 출력합니다.
5. **실험을 실행합니다.**
    - `client.evaluate`에 평가 대상·데이터셋 이름·채점자 목록을 넣어 실행하고, 질문마다 규칙 점수와 모델 점수를 표로 출력합니다.
    - `client.evaluate`가 화면에 출력하는 실험 URL 줄은 `redirect_stdout`으로 잡아 두고, 대신 「실험 URL: (LangSmith 화면에서 확인)」 한 줄을 출력합니다.
    - 실험 이름 접두어는 `library-v1`입니다.
6. **게이트를 판정합니다.**
    - 모델 채점자 점수의 평균을 기준 0.8과 비교해 통과 또는 차단을 출력합니다.


## 5. 코드 골격 — LangSmith 평가 4단

LangSmith로 평가 절차를 세우는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 데이터셋 등록 | 질문과 정답 텍스트를 짝지어 데이터셋으로 올립니다 | `client.create_dataset(...)`, `client.create_examples(...)` | 1 |
| ② 평가 대상 지정 | 입력을 받아 답을 돌려주는 함수 하나를 평가 대상으로 지정합니다 | `def target(inputs: dict) -> dict` | 2 |
| ③ evaluator 정의 | 채점 함수를 만듭니다. 규칙 채점과 모델 채점 두 벌을 씁니다 | `def rule_content(...)`, `with_structured_output(Judge)` | 3, 4 |
| ④ 실행·게이트 | 셋을 넣어 실험을 돌리고, 평균 점수를 기준과 비교해 통과와 차단을 정합니다 | `client.evaluate(target, data=..., evaluators=[...])` | 5, 6 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델을 준비합니다. `warnings.filterwarnings` 두 줄은 추적 라이브러리가 내는 직렬화 경고와 진행 막대 경고를 화면에서 감춥니다. 동작에는 영향이 없습니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- `LANGSMITH_TRACING`을 켜고 프로젝트 이름을 `sesac-lec05-ex02`로 정하면, 아래에서 부르는 모델 호출과 실험이 LangSmith의 그 프로젝트에 남습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import io   # io — io.StringIO()에 client.evaluate가 출력하는 실험 URL 줄을 잡아 둡니다
import os
import warnings
from contextlib import redirect_stdout

from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langsmith import Client
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
warnings.filterwarnings("ignore", message="IProgress not found")

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex02"

# 이 실습에서 부르는 모델 이름
MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
llm = init_chat_model(MODEL, model_provider="litellm")
print("모델 준비를 마쳤습니다. LangSmith 프로젝트:", os.environ["LANGSMITH_PROJECT"])

# 주어진 자료: 골든 데이터셋 GOLDEN(질문·정답 짝)과 모델 채점자에 넣는 지시문 JUDGE_GUIDE — 값을 그대로 씁니다
GOLDEN = [
    {"question": "안녕하세요!",
     "answer": "안녕하세요! 무엇을 도와드릴까요?", "must_know": True},
    {"question": "책은 몇 권까지 며칠 동안 빌릴 수 있나요?",
     "answer": "회원 한 사람이 5권까지 14일 동안 빌릴 수 있습니다.", "must_know": True},
    {"question": "대출 기간을 연장할 수 있나요?",
     "answer": "반납 예정일 전에 홈페이지에서 1회 7일 연장할 수 있습니다. 예약자가 있는 책은 연장할 수 없습니다.", "must_know": True},
    {"question": "도서관 근처에 맛있는 식당이 있나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
    {"question": "파이썬 리스트 정렬은 어떻게 하나요?",
     "answer": "해당 내용은 확인할 수 없습니다.", "must_know": False},
]

JUDGE_GUIDE = (
    "너는 엄격한 채점자다. 평가 대상의 답이 정답(reference)과 부합하는지, "
    "어긋난 지점이 없는지 판정하라."
)


평가 대상이 될 도서관 안내 서비스입니다. 이 서비스는 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. 함수 `answer`가 질문을 받아 FAQ만 근거로 답을 돌려줍니다.


In [ ]:
FAQ = """
[대출] Q: 책은 몇 권까지 며칠 동안 빌릴 수 있나요?
A: 회원 한 사람이 5권까지 14일 동안 빌릴 수 있습니다.
[연장] Q: 대출 기간을 연장할 수 있나요?
A: 반납 예정일 전에 홈페이지에서 1회 7일 연장할 수 있습니다. 예약자가 있는 책은 연장할 수 없습니다.
[휴관] Q: 휴관일은 언제인가요?
A: 매주 월요일과 법정 공휴일에 휴관합니다.
[연체] Q: 반납이 늦으면 어떻게 되나요?
A: 연체한 날수만큼 대출이 정지됩니다. 연체료는 없습니다.
"""

SERVICE_GUIDE = (
    "너는 시립도서관의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
    "인사말에는 짧은 인사로 답한다. "
    "FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.\n"
    "=== FAQ ===\n" + FAQ
)


def answer(question: str) -> str:
    """안내 서비스: 질문을 받아 FAQ만 근거로 답을 돌려준다."""
    res = llm.invoke([("system", SERVICE_GUIDE), ("human", question)])
    return res.content.strip()


print(answer("책은 몇 권까지 며칠 동안 빌릴 수 있나요?"))

### 단계 ① — 데이터셋 등록 (요구사항 1)

LangSmith의 데이터셋은 예시의 모음이고, 예시 하나는 `inputs`(질문)와 `outputs`(기준)의 짝입니다. 채점자는 나중에 이 `outputs`를 `reference_outputs`라는 이름으로 받습니다. 같은 이름의 데이터셋을 다시 만들면 오류가 나므로, 먼저 `has_dataset`으로 있는지 확인하고 있으면 재사용합니다.


In [ ]:
# 여기에 단계 ①(데이터셋 등록 — 이미 있으면 재사용)을 작성합니다.

### 단계 ② — 평가 대상 지정 (요구사항 2)

평가 대상은 `inputs` 딕셔너리를 받아 딕셔너리를 돌려주는 함수 하나입니다. LangSmith가 데이터셋의 질문마다 이 함수를 부르고, 돌려받은 딕셔너리를 채점자의 `outputs`로 넘깁니다. 안내 서비스 자체는 고치지 않고, 서비스를 감싸는 함수만 씁니다.


In [ ]:
# 여기에 단계 ②(평가 대상 함수 target 정의)를 작성합니다.

### 단계 ③ — evaluator 정의 (요구사항 3, 4)

- 채점자는 `inputs`·`outputs`·`reference_outputs` 세 딕셔너리를 받아 `{"key": 이름, "score": 점수}` 딕셔너리를 돌려주는 함수입니다.
- 규칙 채점자 `rule_content`는 모델을 부르지 않고 문자열 검사만으로 점수를 냅니다.
- 모델 채점자 `judge_faithful`은 `Judge` 스키마를 강제한 모델을 한 번 불러 `faithful` 값을 점수로, `reason` 값을 근거(`comment`)로 돌려줍니다.


In [ ]:
# 여기에 단계 ③(규칙 채점자 rule_content, Judge 스키마, 모델 채점자 judge_faithful 정의)을 작성합니다.

### 단계 ④ — 실행·게이트 (요구사항 5, 6)

`client.evaluate`가 데이터셋의 질문마다 평가 대상을 부르고, 세 쌍을 채점자마다 넘겨 점수를 모읍니다. 결과를 돌면서 질문마다 두 점수를 표로 출력하고, 모델 채점자 점수의 평균을 기준과 비교해 통과 또는 차단을 정합니다. `max_concurrency=1`은 질문을 하나씩 차례대로 평가하라는 뜻입니다. `client.evaluate`는 실험 URL을 화면에 직접 출력하므로, `redirect_stdout`으로 그 출력을 잡아 두고 URL 대신 안내 문장을 출력합니다.

`client.evaluate(…, experiment_prefix=접두어)`의 결과 한 건 `r`에서 질문은 `r["example"].inputs["question"]`, 채점 결과 목록은 `r["evaluation_results"]["results"]`(원소마다 `key`·`score`)입니다.


In [ ]:
# 여기에 단계 ④(redirect_stdout으로 감싼 client.evaluate 실행, 점수 표 출력, 게이트 판정)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 ① 출력에 데이터셋 이름 `sesac-lec05-ex02-library`와 「만들고 예시 5개를 올렸습니다」 또는 「이미 있어 재사용합니다」 문장이 출력됩니다.
2. 단계 ③ 출력에서 대출 규정 질문의 규칙 점수와 모델 점수가 모두 1이고, 모델 채점자의 `comment`에 정답과 비교한 근거가 적혀 있습니다.
3. 단계 ④ 표에 질문 5개가 한 줄씩 출력되고, FAQ 밖 질문 2개(식당·파이썬)의 규칙 점수가 1입니다.
4. 마지막 줄에 모델 점수 평균과 기준 0.8, 통과 또는 차단이 출력됩니다. LangSmith 화면의 데이터셋 `sesac-lec05-ex02-library`에 `library-v1`로 시작하는 실험이 새로 생깁니다.

네 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.
